# Phase 165 — CryptoBench Persistence Feature Ablation

Controlled 9-arm × 8-seed ablation using the exact Phase159 graph dataset and the Phase163/164 model and training implementation.

**Primary question:** which persistence-based topological descriptors or homology dimensions provide the predictive information beyond ESM + structure?

**Hypothesis:** the predictive information contributed by TDA is concentrated in specific persistence descriptors and/or homology dimensions rather than being uniformly distributed across all topological features.

The full persistence representation contains **23 features**: 7 H0 + 8 H1 + 8 H2.


## Phase165 design

All arms retain:
- structure features: columns 0:27
- ESM features: columns 27:507
- no ANM-MSF
- no Betti features
- no burial/geometry features

Only the selected persistence columns are unmasked.

### Arms
- `h0_persistence`: 7
- `h1_persistence`: 8
- `h2_persistence`: 8
- `total_max`: 6
- `mean_std`: 6
- `entropy`: 3
- `birth_death`: 5
- `counts`: 3
- `all_persistence`: 23

Total: **9 arms × 8 seeds = 72 trainings**.


In [ ]:
!pip install -q torch-geometric==2.6.1 scipy

In [ ]:
import copy, json, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, roc_auc_score, matthews_corrcoef, f1_score, precision_score, recall_score
from scipy.stats import ttest_rel, wilcoxon

def locate(f):
    hits=[]
    for root in (Path('/kaggle/input'), Path('/kaggle/working'), Path('.')):
        if root.exists(): hits.extend(root.rglob(f))
    if not hits: raise FileNotFoundError(f'Could not find {f}')
    return max(hits, key=lambda p:p.stat().st_size)

blob_path=locate('phase159_cryptobench_anmmsf_graphs.pt')
print('USING:',blob_path)
print('SIZE GB:',blob_path.stat().st_size/1024**3)
blob=torch.load(blob_path,map_location='cpu',weights_only=False)
dataset=blob['dataset']
N_PRE=int(blob['n_pre_topo']); TOPO_DIM=int(blob['topo_dim']); MSF_COL_IDX=int(blob['msf_col_idx']); POS_WEIGHT=float(blob['pos_weight'])
assert N_PRE==508 and TOPO_DIM==34 and MSF_COL_IDX==507
splits={s:[d for d in dataset if d.split==s] for s in ('train','val','test')}
print('node dim:',dataset[0].x.shape[1])
print('splits:',{k:len(v) for k,v in splits.items()})

GPU_OK=False
if torch.cuda.is_available():
    try:
        q=torch.randn(256,256,device='cuda'); (q@q).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print('GPU:',torch.cuda.get_device_name(0),'PASS')
    except Exception as e: print('GPU probe failed:',e)
DEVICE=torch.device('cuda' if GPU_OK else 'cpu')
WORK=Path('/kaggle/working'); (WORK/'cache').mkdir(exist_ok=True)


In [ ]:
class EGCL(nn.Module):
    def __init__(self,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); self.uc=uc
        self.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        self.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        self.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            self.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(self.c[-1].weight,gain=0.001)
        self.norm=nn.LayerNorm(h)
    def forward(self,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=self.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*self.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=self.norm(h+self.n(torch.cat([h,agg/deg],-1)))
        if self.uc:
            w=self.c(m)*rel/(d2.sqrt()+1.0)
            x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(self,h,t):
        super().__init__(); self.p=nn.Linear(t,h); self.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(self,h,t): return h+self.g(torch.cat([h,t],-1))*self.p(t)

class LocalizingEGNN(nn.Module):
    def __init__(self,n_pre,td,h=128,L=4,dr=0.1):
        super().__init__(); self.n_pre=n_pre
        self.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        self.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        self.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        self.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        self.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))
    def forward(self,d):
        h=self.embed(d.x); pos=d.pos; t=d.x[:,self.n_pre:]
        for l,f in zip(self.ls,self.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits=self.head(h).squeeze(-1)
        batch=d.batch if getattr(d,'batch',None) is not None else torch.zeros(h.size(0),dtype=torch.long,device=h.device)
        loc_logit=self.loc_head(h).squeeze(-1); attn=pyg_softmax(loc_logit,batch)
        pred_center=torch.zeros(int(batch.max().item())+1,3,device=h.device).index_add_(0,batch,attn.unsqueeze(-1)*pos)
        return logits,pred_center


In [ ]:
# Each arm always retains ESM + structure and adds ONLY the selected persistence subset.
# ANM-MSF and Betti/burial features are deliberately excluded.
# The full persistence representation contains 23 features: 7 H0 + 8 H1 + 8 H2.

TOPO_NAMES = list(blob['topo_names'])
assert len(TOPO_NAMES) == 34

PERSISTENCE_LOCAL = [
    0,1,2,3,4,5,6,
    10,11,12,13,14,15,16,17,
    21,22,23,24,25,26,27,28
]

# Persistence by homology dimension
H0_PERSISTENCE_LOCAL = list(range(0,7))
H1_PERSISTENCE_LOCAL = list(range(10,18))
H2_PERSISTENCE_LOCAL = list(range(21,29))

# Feature-family subsets
TOTAL_MAX_LOCAL = [1,2,11,12,22,23]
MEAN_STD_LOCAL = [3,4,13,14,24,25]
ENTROPY_LOCAL = [5,15,26]
BIRTH_DEATH_LOCAL = [6,16,17,27,28]
COUNTS_LOCAL = [0,10,21]

def topo_cols(local_indices):
    return [508+i for i in local_indices]

ARMS = {
    'h0_persistence': topo_cols(H0_PERSISTENCE_LOCAL),
    'h1_persistence': topo_cols(H1_PERSISTENCE_LOCAL),
    'h2_persistence': topo_cols(H2_PERSISTENCE_LOCAL),
    'total_max': topo_cols(TOTAL_MAX_LOCAL),
    'mean_std': topo_cols(MEAN_STD_LOCAL),
    'entropy': topo_cols(ENTROPY_LOCAL),
    'birth_death': topo_cols(BIRTH_DEATH_LOCAL),
    'counts': topo_cols(COUNTS_LOCAL),
    'all_persistence': topo_cols(PERSISTENCE_LOCAL),
}

def masked_dataset(ds,arm):
    out=[]
    for d in ds:
        d2=copy.copy(d); d2.x=d.x.clone(); keep=torch.zeros(d2.x.size(1),dtype=torch.bool)
        keep[0:27]=True       # structure
        keep[27:507]=True     # ESM
        keep[ARMS[arm]]=True  # selected persistence features
        d2.x[:,~keep]=0.0
        out.append(d2)
    return out

masked={arm:{s:masked_dataset(splits[s],arm) for s in splits} for arm in ARMS}
for arm in ARMS:
    print(arm, 'persistence dims=',len(ARMS[arm]), {s:len(masked[arm][s]) for s in splits})


## Persistence feature mapping sanity check

The persistence feature indices are derived from the Phase159 `TOPO_NAMES` ordering. The following cell prints every arm and its exact feature names before training.


In [ ]:
for arm, idx in ARMS.items():
    local = [i - 508 for i in idx]
    print(f"{arm:20s} {len(idx):2d} features")
    print([TOPO_NAMES[i] for i in local])


In [ ]:
def focal_loss_with_logits(logits,targets,alpha=0.9,gamma=2.0):
    bce=nn.functional.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    p=torch.sigmoid(logits); pt=p*targets+(1-p)*(1-targets); at=alpha*targets+(1-alpha)*(1-targets)
    return (at*(1-pt).pow(gamma)*bce).mean()

def run_epoch(model,loader,opt=None,lam=0.1):
    train=opt is not None; model.train(train); huber=nn.SmoothL1Loss(beta=4.0); ps=[]; ys=[]; errs=[]
    for b in loader:
        b=b.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits,pc=model(b); loss=focal_loss_with_logits(logits,b.y)+lam*huber(pc,b.true_center.view(-1,3))
        if train:
            opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy()); errs.append((pc-b.true_center.view(-1,3)).norm(dim=1).detach().cpu().numpy())
    p=np.concatenate(ps); y=np.concatenate(ys); e=np.concatenate(errs); pred=p>=0.5
    return {'pr_auc':float(average_precision_score(y,p)),'roc_auc':float(roc_auc_score(y,p)),'mcc':float(matthews_corrcoef(y,pred)),'f1':float(f1_score(y,pred,zero_division=0)),'precision':float(precision_score(y,pred,zero_division=0)),'recall':float(recall_score(y,pred,zero_division=0)),'dcc4':float((e<4).mean()),'mean_center_error':float(e.mean())}

def train_arm(arm,seed,epochs=40,patience=8,batch=16,lam=0.1):
    torch.manual_seed(seed); np.random.seed(seed)
    model=LocalizingEGNN(N_PRE,TOPO_DIM).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4); sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    tl=DataLoader(masked[arm]['train'],batch_size=batch,shuffle=True); vl=DataLoader(masked[arm]['val'],batch_size=batch)
    best=-1; state=None; bad=0
    for ep in range(epochs):
        run_epoch(model,tl,opt,lam); v=run_epoch(model,vl,lam=lam); sch.step()
        if v['pr_auc']>best: best=v['pr_auc']; state={k:t.detach().cpu().clone() for k,t in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=patience: break
    model.load_state_dict(state); model.eval(); val=run_epoch(model,vl,lam=lam); te=run_epoch(model,DataLoader(masked[arm]['test'],batch_size=batch),lam=lam)
    path=WORK/f'cache/phase165_{arm}_seed{seed}.pt'; torch.save(model.state_dict(),path)
    return val,te,str(path)


## Run nine persistence arms × eight seeds

This is **72 trainings**. For a pilot only, temporarily use `SEEDS=list(range(3))`; restore eight seeds for the final result.


In [ ]:
SEEDS=list(range(8)); results=[]; t0=time.time()
for arm in ARMS:
    for seed in SEEDS:
        val,te,path=train_arm(arm,seed)
        row={'arm':arm,'seed':seed,'val':val,'test':te,'checkpoint':path}; results.append(row)
        print(f"{arm:14s} seed={seed} val_PR={val['pr_auc']:.4f} val_ROC={val['roc_auc']:.4f} test_PR={te['pr_auc']:.4f} test_ROC={te['roc_auc']:.4f} MCC={te['mcc']:.4f} DCC4={te['dcc4']:.3f} elapsed={(time.time()-t0)/60:.1f}m")
json.dump(results,open(WORK/'phase165_tda_component_results.json','w'),indent=2)


In [ ]:
import pandas as pd
rows=[]
for r in results:
    rows.append({'arm':r['arm'],'seed':r['seed'],**{f'val_{k}':v for k,v in r['val'].items()},**{f'test_{k}':v for k,v in r['test'].items()}})
df=pd.DataFrame(rows)
df.to_csv(WORK/'phase165_tda_component_seed_results.csv',index=False)
summary=df.groupby('arm').agg({
    'seed':['count'],
    'val_pr_auc':['mean','std'],
    'test_pr_auc':['mean','std'],
    'test_roc_auc':['mean','std'],
    'test_mcc':['mean','std'],
    'test_f1':['mean','std'],
    'test_precision':['mean','std'],
    'test_recall':['mean','std'],
    'test_dcc4':['mean','std'],
    'test_mean_center_error':['mean','std'],
}).reset_index()
summary.columns=['arm','n_seeds','val_pr_auc_mean','val_pr_auc_std','test_pr_auc_mean','test_pr_auc_std','test_roc_auc_mean','test_roc_auc_std','test_mcc_mean','test_mcc_std','test_f1_mean','test_f1_std','test_precision_mean','test_precision_std','test_recall_mean','test_recall_std','test_dcc4_mean','test_dcc4_std','test_center_error_mean','test_center_error_std']
summary.to_csv(WORK/'phase165_tda_component_summary.csv',index=False)
print(summary.sort_values('val_pr_auc_mean',ascending=False).to_string(index=False))


## Paired significance tests

Same eight seeds are paired across arms. Primary comparison: `all_tda` versus each reduced TDA representation on validation PR-AUC.


In [ ]:
def vec(arm,split='val',metric='pr_auc'):
    rr=sorted([r for r in results if r['arm']==arm],key=lambda x:x['seed'])
    return np.array([r[split][metric] for r in rr],float)

sig=[]
for arm in ARMS:
    if arm=='all_tda': continue
    a=vec('all_tda'); b=vec(arm); tt=ttest_rel(a,b)
    try: wp=wilcoxon(a,b).pvalue
    except Exception: wp=float('nan')
    sig.append({'comparison':'all_tda - '+arm,'metric':'val_pr_auc','all_tda_mean':float(a.mean()),'arm_mean':float(b.mean()),'mean_delta':float((a-b).mean()),'paired_t':float(tt.statistic),'t_p':float(tt.pvalue),'wilcoxon_p':float(wp)})
sig_df=pd.DataFrame(sig)
print(sig_df.to_string(index=False))
sig_df.to_csv(WORK/'phase165_tda_component_paired_statistics.csv',index=False)
json.dump(sig,open(WORK/'phase165_tda_component_paired_statistics.json','w'),indent=2)


## Interpretation

The key result is whether compact persistence subsets match the performance of `all_persistence`.

Interpret the results by asking:
1. Which homology dimension (H0, H1, H2) is most predictive?
2. Which descriptor family (magnitude, distribution, entropy, birth/death timing, counts) carries the strongest signal?
3. Does the full 23-feature persistence representation provide a meaningful advantage over compact subsets?

Do not claim that a subset is superior to `all_persistence` unless the paired statistical tests support that conclusion. These seed-level comparisons are useful for ablation evidence, while final uncertainty claims should use chain-level/bootstrap analyses where appropriate.
